In [ ]:
!nvidia-smi

In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

!pip install ninja

In [ ]:
import torch
from torch.utils.cpp_extension import load_inline


cuda_source = """
#include <torch/extension.h>

#define TILE_WIDTH 16

__global__
void matrixMult(int *A, int *B, int *C, int inner_dim, int w, int h)
{
    int col = blockDim.x * blockIdx.x + threadIdx.x;
    int row = blockDim.y * blockIdx.y + threadIdx.y;

    // we need something to know the future m*n
    // we need to know the inner dimesion of the matrix a
    if (col < w && row < h)
    {
        int sum = 0;
        for (int i = 0; i < inner_dim; i++) {
            sum += A[row*w +i] * B[i*w + col];
        }

        C[row * w + col] = sum;
    }
}

void run_matrixmul(torch::Tensor A, torch::Tensor B, torch::Tensor C, int width) {
  // Extract the raw C++ pointers from the PyTorch tensors
  int* A_ptr = A.data_ptr<int>();
  int* B_ptr = B.data_ptr<int>();
  int* C_ptr = C.data_ptr<int>();

  // Set up your blocks and threads
  dim3 threads(TILE_WIDTH, TILE_WIDTH);
  
  int blockX = (width + TILE_WIDTH - 1) / TILE_WIDTH;
  int blockY = (width + TILE_WIDTH - 1) / TILE_WIDTH;
  dim3 blocks(blockX, blockY);

  // Launch!
  matrixMult<<<blocks, threads >>>(A_ptr, B_ptr, C_ptr, width, width, width);
}


__global__ 
void matmul(int *A, int *B, int *C, int width)
{
    // get y and x
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // create shared memory variable 2d array
    __shared__ int m_ds[TILE_WIDTH][TILE_WIDTH];
    __shared__ int n_ds[TILE_WIDTH][TILE_WIDTH];

    int sum = 0;
    // make the loop for width of the array / TILE_WIDTH
    for (int i = 0; i * TILE_WIDTH < width; i++)
    {
        // load m and n into shared memory
        m_ds[threadIdx.y][threadIdx.x] = A[row * width + (i * TILE_WIDTH + threadIdx.x)];
        n_ds[threadIdx.y][threadIdx.x] = B[(i * TILE_WIDTH + threadIdx.y) * width + col];
        // sync threads
        __syncthreads();
        // from i to TILE_WIDTH, load all the numbers that other threads have loaded into shared memory
        for (int j = 0; j < TILE_WIDTH; j++)
        {
            // mutltiple m[i] n[i] then add to the sum
            sum += m_ds[threadIdx.y][j] * n_ds[j][threadIdx.x];
        }
        // sync threads
        __syncthreads();
    }
    // load the sum at C[x][y]
    C[row *width + col] = sum;
}

void run_matmul(torch::Tensor A, torch::Tensor B, torch::Tensor C, int width) {
  // Extract the raw C++ pointers from the PyTorch tensors
  int* A_ptr = A.data_ptr<int>();
  int* B_ptr = B.data_ptr<int>();
  int* C_ptr = C.data_ptr<int>();

  // Set up your blocks and threads
  dim3 threads(TILE_WIDTH, TILE_WIDTH);
  
  int blockX = (width + TILE_WIDTH - 1) / TILE_WIDTH;
  int blockY = (width + TILE_WIDTH - 1) / TILE_WIDTH;
  dim3 blocks(blockX, blockY);

  // Launch!
  matmul<<<blocks, threads >>>(A_ptr, B_ptr, C_ptr, width);
}
"""

cpp_source="""
void run_matmul(torch::Tensor A, torch::Tensor B, torch::Tensor C, int width);

void run_matrixmul(torch::Tensor A, torch::Tensor B, torch::Tensor C, int width);
"""

cuda_module = load_inline(
    name='custom_matmul_v3',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['run_matmul', 'run_matrixmul'],
    verbose=True
)

N=1024

A_tensor = torch.randint(0,10, (N,N), dtype=torch.int32, device='cuda')
B_tensor = torch.randint(0,10, (N,N), dtype=torch.int32, device='cuda')
C_tensor = torch.zeros((N,N), dtype=torch.int32, device='cuda')

# cuda_module.run_matmul(A_tensor, B_tensor, C_tensor, N)

A_float = A_tensor.float()
B_float = B_tensor.float()

print("PyTorch Speed:")
%timeit torch.matmul(A_float, B_float)

print("Custom Kernel No tiling Speed:")
%timeit cuda_module.run_matrixmul(A_tensor, B_tensor, C_tensor, N); torch.cuda.synchronize()


print("Custom Kernel tiling Speed:")
%timeit cuda_module.run_matmul(A_tensor, B_tensor, C_tensor, N); torch.cuda.synchronize()

# Correctness Check
expected_C = torch.matmul(A_float, B_float).to(torch.int32)

is_correct = torch.equal(C_tensor, expected_C)
print(f"Kernel accurate: {is_correct}")